## Report
```
The input protein string (up to 1000 aa) is transferred to the GPU as a char array.
Monoisotopic masses for all 20 amino acids are stored in a __constant__ array indexed by letter offset (char - 'A'),
copied to the device before the kernel launch.
A CUDA kernel assigns one thread per amino acid and each thread performs a global atomicAdd on a single double accumulator using the mass corresponding to its residue.
The result is copied back to the host and printed to three decimal places.
```

In [1]:
%%writefile calculating_protein_mass.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>

#define PROTEIN_INPUT "SKADYEK"

__constant__ double d_masses[26];

__global__ void calculating_protein_mass_kernel(const char *seq, double *total, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        atomicAdd(total, d_masses[seq[idx] - 'A']);
    }
}

void launch_protein_kernel(const char *h_seq, int n)
{
    static const double h_masses[26] = {
         71.03711,  /* A */
          0.0,      /* B */
        103.00919,  /* C */
        115.02694,  /* D */
        129.04259,  /* E */
        147.06841,  /* F */
         57.02146,  /* G */
        137.05891,  /* H */
        113.08406,  /* I */
          0.0,      /* J */
        128.09496,  /* K */
        113.08406,  /* L */
        131.04049,  /* M */
        114.04293,  /* N */
          0.0,      /* O */
         97.05276,  /* P */
        128.05858,  /* Q */
        156.10111,  /* R */
         87.03203,  /* S */
        101.04768,  /* T */
          0.0,      /* U */
         99.06841,  /* V */
        186.07931,  /* W */
          0.0,      /* X */
        163.06333,  /* Y */
          0.0       /* Z */
    };

    cudaMemcpyToSymbol(d_masses, h_masses, 26 * sizeof(double));

    char *d_seq = NULL;
    double *d_total = NULL;

    cudaMalloc((void **)&d_seq, n * sizeof(char));
    cudaMalloc((void **)&d_total, sizeof(double));

    cudaMemset(d_total, 0, sizeof(double));
    cudaMemcpy(d_seq, h_seq, n * sizeof(char), cudaMemcpyHostToDevice);

    int block_size = 256;
    int grid_size = (n + block_size - 1) / block_size;

    calculating_protein_mass_kernel<<<grid_size, block_size>>>(d_seq, d_total, n);

    cudaDeviceSynchronize();

    double h_total = 0.0;

    cudaMemcpy(&h_total, d_total, sizeof(double), cudaMemcpyDeviceToHost);

    printf("%.3f\n", h_total);

    cudaFree(d_seq);
    cudaFree(d_total);
}

int main(void)
{
    const char *protein = PROTEIN_INPUT;
    int n = (int)strlen(protein);

    launch_protein_kernel(protein, n);

    return 0;
}

Writing calculating_protein_mass.cu


In [2]:
!nvcc -arch=sm_75 calculating_protein_mass.cu -o calculating_protein_mass

In [3]:
!./calculating_protein_mass

821.392
